# Shared-Kernel GP Interpolator

Multi-output surrogate modeling using a Gaussian process with a shared Matern 5/2 kernel
and a degree-2 polynomial mean. Instead of training independent GPs per output pixel,
we use one kernel matrix for all outputs — a single Cholesky factorization + matrix solve.

## Installation

```bash
# requires python 3.13+, uv (https://docs.astral.sh/uv/)
uv sync

# Jetson Orin: the stock PyPI jaxlib lacks SM 8.7 CUDA kernels.
# Build from source with HERMETIC_CUDA_COMPUTE_CAPABILITIES=sm_87,
# or set JAX_PLATFORMS=cpu to run on CPU (still fast for this problem size).
```

In [ ]:
import os
os.environ["XLA_FLAGS"] = (
    "--xla_gpu_autotune_level=0 --xla_gpu_enable_triton_gemm=false"
)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import time
from scipy.interpolate import RBFInterpolator as ScipyRBF

print(f"Device: {jax.devices()}")

## Kernel and basis functions

Matern 5/2 with isotropic lengthscale, plus a degree-2 polynomial basis
(constant + linear + quadratic/cross terms) for the universal kriging mean.

In [ ]:
def _poly_basis(X, degree):
    """Monomial basis up to given degree. Returns (n, p) matrix."""
    n = X.shape[0]
    cols = [jnp.ones((n, 1))]
    if degree >= 1:
        cols.append(X)
    if degree >= 2:
        m = X.shape[1]
        for i in range(m):
            for j in range(i, m):
                cols.append((X[:, i] * X[:, j])[:, None])
    return jnp.concatenate(cols, axis=1)


def _matern52_K(X1, X2, log_ls, log_sv):
    """Matern 5/2 kernel matrix between X1 (n1, m) and X2 (n2, m)."""
    ls = jnp.exp(log_ls)
    sv = jnp.exp(log_sv)
    diff = X1[:, None, :] / ls - X2[None, :, :] / ls
    r2 = jnp.sum(diff**2, axis=-1)
    r = jnp.sqrt(jnp.maximum(r2, 1e-20))
    s5r = jnp.sqrt(5.0) * r
    return sv * (1.0 + s5r + 5.0 / 3.0 * r2) * jnp.exp(-s5r)

## Interpolator class

All outputs share one kernel matrix (since K depends only on input points).
Fitting is a single Cholesky factorization + universal kriging solve for all
output dimensions simultaneously.

In [ ]:
class SharedGPInterpolator:
    """GP interpolator with a shared Matern 5/2 kernel across all outputs.

    Parameters
    ----------
    parameters : array (n, m) — training input points
    data : array (n, *output_shape) — training outputs (any shape per sample)
    poly_degree : int — polynomial mean degree (0, 1, or 2)
    lengthscale : float — isotropic Matern 5/2 lengthscale (in normalized input space)
    signal_variance : float — kernel signal variance
    noise_variance : float — diagonal nugget for numerical stability
    """

    def __init__(self, parameters, data, *, poly_degree=2, lengthscale=2.5,
                 signal_variance=0.08, noise_variance=1e-6):
        X = jnp.asarray(parameters, dtype=jnp.float32)
        if X.ndim == 1:
            X = X[:, None]
        data = jnp.asarray(data, dtype=jnp.float32)

        # normalize inputs to zero mean, unit variance
        self._X_mean = jnp.mean(X, axis=0)
        self._X_std = jnp.std(X, axis=0)
        Xn = (X - self._X_mean) / self._X_std

        self.X_train = Xn
        self._output_shape = data.shape[1:]
        self._poly_degree = poly_degree
        n, m = Xn.shape
        self._m = m
        D = int(np.prod(self._output_shape))
        Y = data.reshape(n, D)
        H = _poly_basis(Xn, poly_degree)

        # fixed hyperparameters (log-space)
        self._log_ls = jnp.full(m, jnp.log(lengthscale))
        self._log_sv = jnp.log(jnp.float32(signal_variance))
        log_nv = jnp.log(jnp.float32(noise_variance))

        # one Cholesky, solve all outputs at once
        t0 = time.perf_counter()
        K = _matern52_K(Xn, Xn, self._log_ls, self._log_sv)
        K = K + (jnp.exp(log_nv) + 1e-6) * jnp.eye(n)
        L = jnp.linalg.cholesky(K)

        # universal kriging solve
        R = jax.scipy.linalg.solve_triangular(L, H, lower=True)
        V = jax.scipy.linalg.solve_triangular(L, Y, lower=True)
        RtR_L = jnp.linalg.cholesky(R.T @ R)
        self._Beta = jax.scipy.linalg.cho_solve((RtR_L, True), R.T @ V)  # (p, D)
        self._Alpha = jax.scipy.linalg.cho_solve(
            (L, True), Y - H @ self._Beta)                                # (n, D)
        self._Alpha.block_until_ready()
        dt = time.perf_counter() - t0

        # reconstruction error at training points
        K_nf = _matern52_K(Xn, Xn, self._log_ls, self._log_sv)
        Y_hat = K_nf @ self._Alpha + H @ self._Beta
        rel_err = float(jnp.linalg.norm(Y_hat - Y) / jnp.linalg.norm(Y))

        print(f"Fit {D} outputs in {dt:.2f}s  |  "
              f"ls={lengthscale}  poly_degree={poly_degree}  "
              f"train_recon_err={rel_err:.2e}")

    def predict(self, x):
        """Predict at a single point. x: (m,)"""
        xn = (x - self._X_mean) / self._X_std
        k = _matern52_K(xn[None, :], self.X_train,
                        self._log_ls, self._log_sv).squeeze(0)
        h = _poly_basis(xn[None, :], self._poly_degree).squeeze(0)
        return (k @ self._Alpha + h @ self._Beta).reshape(self._output_shape)

    def predict_batch(self, X):
        """Predict at multiple points. X: (batch, m)"""
        Xn = (X - self._X_mean) / self._X_std
        K_new = _matern52_K(Xn, self.X_train, self._log_ls, self._log_sv)
        H_new = _poly_basis(Xn, self._poly_degree)
        return (K_new @ self._Alpha + H_new @ self._Beta).reshape(
            -1, *self._output_shape)

## Synthetic training data

A 64x32 spectrogram-like field parameterized by 3 scalars (amplitude, envelope position, spectral shift). 5x5x5 = 125 training samples on a grid.

In [ ]:
n_time, n_freq = 64, 32
times = np.linspace(0, 1, n_time)
freqs = np.linspace(50, 100, n_freq)

def ground_truth(theta):
    t1, t2, t3 = theta
    amplitude = t1 + 0.3 * t2 * t3
    envelope = np.exp(-((times - 0.5 * t2) ** 2) / (0.1 + 0.05 * t3))
    spectrum = np.sinc((freqs - 70 * t1) / (10 + 2 * t2))
    return amplitude * envelope[:, None] * spectrum[None, :]

train_params = np.array(np.meshgrid(
    np.linspace(0.8, 1.2, 5),
    np.linspace(0.5, 1.5, 5),
    np.linspace(0.2, 0.8, 5), indexing="ij")).reshape(3, -1).T
train_data = np.stack([ground_truth(p) for p in train_params])
print(f"{train_params.shape[0]} training samples, output shape {train_data.shape[1:]}")

## Fit both interpolators

In [ ]:
gp = SharedGPInterpolator(train_params, train_data,
                          poly_degree=2, lengthscale=2.5)

t0 = time.perf_counter()
rbf = ScipyRBF(train_params, train_data.reshape(len(train_params), -1),
               kernel="cubic", degree=1)
print(f"Scipy RBF fit: {time.perf_counter() - t0:.2f}s")

## Accuracy comparison

Relative L2 error on 50 random test points drawn from within the training domain.

In [ ]:
rng = np.random.default_rng(0)
n_test = 50
test_points = np.column_stack([
    rng.uniform(0.85, 1.15, n_test),
    rng.uniform(0.6, 1.4, n_test),
    rng.uniform(0.25, 0.75, n_test),
])

gp_errs, rbf_errs = [], []
for tp in test_points:
    tr = ground_truth(tp)
    gp_pred = np.array(gp.predict(jnp.array(tp, dtype=jnp.float32)))
    gp_errs.append(np.linalg.norm(gp_pred - tr) / np.linalg.norm(tr))
    rbf_pred = rbf(tp[None]).reshape(n_time, n_freq)
    rbf_errs.append(np.linalg.norm(rbf_pred - tr) / np.linalg.norm(tr))

print(f"{'':15s} {'median':>10s} {'max':>10s}")
print(f"{'GP':15s} {np.median(gp_errs):10.2e} {np.max(gp_errs):10.2e}")
print(f"{'Scipy RBF':15s} {np.median(rbf_errs):10.2e} {np.max(rbf_errs):10.2e}")
print(f"\nGP / RBF ratio:  {np.median(gp_errs)/np.median(rbf_errs):.2f}x")

## Timing comparison

In [ ]:
test_theta = jnp.array([1.05, 0.9, 0.5], dtype=jnp.float32)
test_np = np.array(test_theta)[None, :]

# JIT warmup
predict_jit = jax.jit(gp.predict)
predict_batch_jit = jax.jit(gp.predict_batch)
_ = predict_jit(test_theta).block_until_ready()

# single point
n_trials = 1000
t0 = time.perf_counter()
for _ in range(n_trials):
    predict_jit(test_theta).block_until_ready()
gp_us = (time.perf_counter() - t0) / n_trials * 1e6

t0 = time.perf_counter()
for _ in range(n_trials):
    rbf(test_np)
rbf_us = (time.perf_counter() - t0) / n_trials * 1e6

print(f"Single point:    GP {gp_us:.0f} us   RBF {rbf_us:.0f} us")

# batch
rng2 = np.random.default_rng(1)
n_batch = 2000
batch_np = np.column_stack([
    rng2.uniform(0.85, 1.15, n_batch),
    rng2.uniform(0.6, 1.4, n_batch),
    rng2.uniform(0.25, 0.75, n_batch),
])
batch_jax = jnp.array(batch_np, dtype=jnp.float32)
_ = predict_batch_jit(batch_jax).block_until_ready()

t0 = time.perf_counter()
for _ in range(100):
    predict_batch_jit(batch_jax).block_until_ready()
gp_ms = (time.perf_counter() - t0) / 100 * 1e3

t0 = time.perf_counter()
for _ in range(100):
    rbf(batch_np)
rbf_ms = (time.perf_counter() - t0) / 100 * 1e3

print(f"Batch (2000 pt): GP {gp_ms:.2f} ms   RBF {rbf_ms:.2f} ms")

## Visual comparison

Ground truth vs GP vs RBF for a single test point.

In [ ]:
tp = test_points[0]
tr = ground_truth(tp)
gp_pred = np.array(gp.predict(jnp.array(tp, dtype=jnp.float32)))
rbf_pred = rbf(tp[None]).reshape(n_time, n_freq)

vmin, vmax = tr.min(), tr.max()

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, img, title in zip(axes, [tr, gp_pred, rbf_pred, gp_pred - tr],
                            ["Ground truth", "GP prediction", "Scipy RBF", "GP error"]):
    if title == "GP error":
        im = ax.imshow(img, aspect="auto", cmap="RdBu_r",
                       vmin=-0.05 * (vmax - vmin), vmax=0.05 * (vmax - vmin))
    else:
        im = ax.imshow(img, aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("freq bin")
    ax.set_ylabel("time bin")
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle(f"theta = ({tp[0]:.2f}, {tp[1]:.2f}, {tp[2]:.2f})", y=1.02)
plt.tight_layout()

## Error distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
bins = np.linspace(0, max(max(gp_errs), max(rbf_errs)) * 1.1, 20)
ax.hist(gp_errs, bins=bins, alpha=0.7, label=f"GP (median={np.median(gp_errs):.1e})")
ax.hist(rbf_errs, bins=bins, alpha=0.7, label=f"RBF (median={np.median(rbf_errs):.1e})")
ax.set_xlabel("Relative L2 error")
ax.set_ylabel("Count")
ax.legend()
ax.set_title("Error distribution over 50 test points")
plt.tight_layout()